<a href="https://colab.research.google.com/github/elope237/CAHSI-26research/blob/main/Grover's_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Grovers Search Activity

In [1]:
# @title Initial Set Up

import ipywidgets as widgets
from IPython.display import display

def ask_mcq(qid, question, options, correct):
    radio = widgets.RadioButtons(
        options=options,
        layout=widgets.Layout(width='auto'),
        style={'description_width': '0px'}
    )

    radio.add_class("big-radio")

    btn = widgets.Button(description="Check answer", button_style='primary')
    out = widgets.Output()

    def check_answer(b):
        with out:
            out.clear_output()

            if radio.value == correct:
                print("✅ Correct!")
            else:
                print(f"❌ Incorrect, try again")

    btn.on_click(check_answer)

    display(widgets.HTML(f"<b style='font-size:24px'>{question}</b>"))
    display(radio, btn, out)

    display(widgets.HTML("""
    <style>
    .big-radio label {
        font-size: 20px !important;
    }
    </style>
    """))



## Classical Search

Classical search is a foundational idea in computer science given a clearly defined problem and a set of possible candidates (items in a list, positions in an array, states in a game), a search algorithm’s job is to find a target item/state or decide it isn’t there. Different searches trade off speed and memory to solve a problem.

**Brute force search**
Brute force search (often called linear search) is the simplest approach it starts at the beginning and check each option one by one until you find the target or reach the end.
• When it works best: unsorted data, very small inputs, or when you can’t rely on any structure.

• Time complexity: O(n)

**Binary search**

Binary search is faster, but it requires a key assumption: the data must be sorted. It repeatedly cuts the search range in half by comparing the middle element to the target.

• When it works best: sorted arrays/lists where random access is easy.

• Time complexity: O(log n)

Note: sorting itself can cost time (typically O(n log n)), so binary search is most beneficial when the data is already sorted, or you do many searches after sorting once.

**Hash-based search**

Hash search uses a hash table (like Python dictionaries or Java HashMap). A hash function converts a key into an index so the algorithm can jump directly to the likely location.

• When it works best: when the dataset is large, not sorted, and you will perform many repeated searches.

• Time complexity: O(1)

In [ ]:
# @title Illustration of Binary Search
import matplotlib.pyplot as plt


def binary_search(arr, target):
    """
    Returns (index, steps).
    index = -1 if not found.
    steps = list of (lo, mid, hi) for visualization.
    """
    lo, hi = 0, len(arr) - 1
    steps = []

    while lo <= hi:
        mid = (lo + hi) // 2
        steps.append((lo, mid, hi))

        if arr[mid] == target:
            return mid, steps
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1

    return -1, steps


def plot_binary_search_steps(arr, target, steps, title="Binary Search Midpoint Selection"):
    # if no steps, nothing to plot
    if not steps:
        print("No steps to plot.")
        return

    x = list(range(len(arr)))
    y = [0] * len(arr)

    # one subplot per step, shared on x-axis
    fig, axes = plt.subplots(len(steps), 1, figsize=(10, max(2, 1.4 * len(steps))), sharex=True)
    if len(steps) == 1:
        axes = [axes]

    for i, (lo, mid, hi) in enumerate(steps):
        ax = axes[i]
        # base dots
        ax.scatter(x, y, s=60)
        # highlight mid
        ax.scatter([mid], [0], s=180, marker="o", label=f"mid={mid} (val={arr[mid]})")
        # highlight range
        ax.axvspan(lo - 0.5, hi + 0.5, alpha=0.15, label=f"search range [{lo},{hi}]")
        # axis
        ax.set_yticks([])
        ax.set_xlim(-0.5, len(arr) - 0.5)
        ax.set_title(f"Step {i+1}: lo={lo}, mid={(lo+hi)//2}, hi={hi}")

        # display values within the current search range
        for xi in range(lo, hi + 1):
            ax.text(xi, 0.02, str(arr[xi]), ha="center", va="bottom", fontsize=9)

        # move legend outside the axes on the right
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), borderaxespad=0.)

    # title and layout adjustments
    fig.suptitle(f"{title}\nTarget={target}", y=0.98, fontsize=14)
    # make room on the right for the external legends
    fig.subplots_adjust(right=0.82, top=0.94, hspace=0.6)

    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    # demo array and search
    arr = list(range(0, 33, 3))  # [0,3,6,...,30]
    idx, steps = binary_search(arr, 21)
    plot_binary_search_steps(arr, 21, steps)
    print("found index:", idx)


In [ ]:
# @title Plot of classical search runtimes
import math
import matplotlib.pyplot as plt


# Linear search (average-case)
def steps_linear_avg(n: int) -> float:
    return (n + 1) / 2

# Binary search (average-case)
def steps_binary_avg(n: int) -> int:
    return math.ceil(math.log2(n)) if n > 1 else 1

# Hash table (Python dict) average-case
def steps_hash_avg(_: int) -> int:
    return 1  # average O(1)


def main():
    Ns = [10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000, 300_000, 1_000_000]

    # ---- Ledger (table) ----
    ledger = [
        {"Method": "Brute force (linear search)", "Big-O": "O(n)"},
        {"Method": "Binary search",               "Big-O": "O(log n)"},
        {"Method": "Hash dict lookup",            "Big-O": "O(1)"},
    ]

    print("\nLedger:")
    print(f"{'Method':30} | {'Big-O':8}")
    print("-" * 43)
    for row in ledger:
        print(f"{row['Method'][:30]:30} | {row['Big-O']:8}")

    # ---- Compute average step curves ----
    lin_avg = [steps_linear_avg(n) for n in Ns]
    bin_avg = [steps_binary_avg(n) for n in Ns]
    hsh_avg = [steps_hash_avg(n) for n in Ns]

    # ---- Plot ----
    plt.figure(figsize=(9, 5))
    plt.plot(Ns, lin_avg, marker="o", label="Brute force (linear search) O(n)")
    plt.plot(Ns, bin_avg, marker="o", label="Binary search O(log n)")
    plt.plot(Ns, hsh_avg, marker="o", label="Hash dict O(1)")

    plt.xscale("log")
    #plt.xlim(1, max(Ns))
    plt.yscale("log")
    plt.minorticks_off()  # REMOVE mid / minor tick markers
    plt.xlabel("N (number of size)")
    plt.ylabel("Steps (comparisons)")
    plt.title("Steps vs N")
    plt.grid(True, alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()

In [ ]:
# @title Question 1
ask_mcq(
    "q1",
    "Checking every item one by one is called:",
    ["Binary search", "Brute-force (linear) search", "Random search"],
    correct="Brute-force (linear) search"
)


In [ ]:
# @title Question 2
ask_mcq(
    "q2",
    "Binary search works best when the list is:",
    ["Random", "Sorted", "Very large"],
    correct="Sorted"
)


In [ ]:
# @title Question 3
ask_mcq(
    "q3",
    "What does a Python dictionary search do?",
    ["Look up values quickly", "Sort numbers", "Draw graphs"],
    correct = "Look up values quickly"
)


## Quantum Background

Quantum Computing works by applying "gates" on "qubits".
The typical way of representing qubits is as state vectors.

Lets say we have a qubit, it can either be $|0\rangle$ or $|1\rangle$ analogous to the binary bits 0 and 1

The $|\rangle$ notation is called a "ket" and is just a way to represent column state vectors commonly used in quantum mechanics

The most common kets in quantum computing are $|0\rangle$ = $\begin{pmatrix} 1 \\ 0 \end{pmatrix}$ and $|1\rangle$ = $\begin{pmatrix} 0 \\ 1 \end{pmatrix}$


In classical computing a bit either has an 100% chance to be 0 or an 100% chance to be 1 but in quantum computing the probabilites are not always 100%


For example if we have a generic qubit $|\psi\rangle$ we can represent its state as  $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$


$\alpha$ is the amplitude of the  $|0\rangle$ state and $||\alpha\|^2$ is the probability we measure the qubit to be in the $|0\rangle$ state.


Similarly $\beta$ is the amplitude of the $|1\rangle$ state and $||\beta\|^2$ is the probability we measure the qubit to be in the $|1\rangle$ state.


This also means that $||\alpha\|^2$ + $||\beta\|^2$ must equal 1


For example if $|\psi\rangle$ = $\frac{1}{\sqrt{2}} |0\rangle$ + $\frac{1}{\sqrt{2}} |1\rangle$ then when we measure the qubit we have a 50% chance we will measure the result $|0\rangle$ and a 50% we will measure the result $|1\rangle$

This is called **superposition** where the qubit could either be in the 0 state or the 1 state. This chance to be in either state is one of the most important and powerful ideas of quantum computing.


Conceptual Checkin:

Given the state $|\psi\rangle$ = $\frac{1}{\sqrt{3}} |0\rangle$ + $\sqrt{\frac{2}{3}} |1\rangle$

what is the probability we measure the qubit to be $|0\rangle$  and whats the probability we measure it to be $|1\rangle$  ?

<details>
  <summary>Click to expand the answer<br></summary>

Theres a $\left|\frac{1}{\sqrt{3}}\right|^2$ = $\frac{1}{3}$ chance to measure $|0\rangle$ and a $\left|\sqrt{\frac{2}{3}}\right|^2$ = $\frac{2}{3}$ chance to measure $|1\rangle$


</details>

Now that we know a little bit about what qubits are and how they are represented, lets talk a little bit about Quantum Gates. There are 2 common simple gates we will use in this activity the $X$ (NOT) gate and the Hadamard $H$ gate. There are many other quantum gates but they are either much more complicated or not commonly used in the activity.


The $X$ or NOT gate works just like the NOT gate works in classical computing, it flips 0 to 1 and vice versa.
Its matrix representation is $X = \begin{pmatrix} 0&1 \\ 1&0 \end{pmatrix}$

For example, when applied to the $|0\rangle$ state

$X |0\rangle$ = $\begin{pmatrix} 0&1 \\ 1&0 \end{pmatrix}$  $\begin{pmatrix} 1 \\ 0 \end{pmatrix}$ = $\begin{pmatrix} 0 \\ 1 \end{pmatrix}$= $|1\rangle$

The Hadamard or H gate puts a qubit in equal superposition of its possible states

$H = \frac{1}{\sqrt{2}}\begin{pmatrix}1 & 1 \\ 1 & -1\end{pmatrix}$

For example, when applied to the $|0\rangle$ state

$H |0\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}1 & 1 \\ 1 & -1\end{pmatrix}$ $\begin{pmatrix} 1 \\ 0 \end{pmatrix}$ =
$\frac{1}{\sqrt{2}}\begin{pmatrix}1\\ 1\end{pmatrix}$ =
$\frac{1}{\sqrt{2}}\begin{pmatrix}1\\ 0\end{pmatrix}$ + $\frac{1}{\sqrt{2}}\begin{pmatrix}0\\ 1\end{pmatrix}$ =
$\frac{1}{\sqrt{2}} |0\rangle + \frac{1}{\sqrt{2}} |1\rangle$


$H |1\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix}1 & 1 \\ 1 & -1\end{pmatrix}$ $\begin{pmatrix} 0 \\ 1 \end{pmatrix}$ =
$\frac{1}{\sqrt{2}}\begin{pmatrix}1\\ -1\end{pmatrix}$ =
$\frac{1}{\sqrt{2}}\begin{pmatrix}1\\ 0\end{pmatrix}$ - $\frac{1}{\sqrt{2}}\begin{pmatrix}0\\ 1\end{pmatrix}$ =
$\frac{1}{\sqrt{2}} |0\rangle - \frac{1}{\sqrt{2}} |1\rangle$

In both cases appling the Hadamard gate puts the qubit into a superposition between the two possible states. Because of how powerful this is, we will often start circuits by applying Hadamard gates to qubits.

Usually in a quantum circuit we have multiple qubit systems. In this activity we will mostly work in 2 or 3 qubit systems. If we have 3 qubits with the "0th" qubit in the 1 state, the "1st" qubit in the 1 state and the "2nd" qubit in the 0 state we can represent it using one ket. $|011\rangle$
This is very simular to classical bits.

When we measure the qubit it will collapse into one of the states and will always be in that state when we measure it. This means the qubit loses its special quantum properties so we usually don't measure it until the very end after the circuit is finished.

Another important phyiscs property that quantum computing uses is **interference**.

Imagine dropping two pebbles into water. The ripples spread out as waves, and when they meet, they combine: peaks with peaks make a bigger wave, while a peak meeting a trough cancels out. Quantum particles behave similarly — their probability amplitudes act like waves, so different paths can reinforce each other or cancel out, which is called quantum interference.


In quantum mechanics qubits are often represented as sin or cos waves which means they can constructivly or destructively interfere with eachother. This is **Interference** and it is another property of quantum computers that make them so powerful.

The code below is a way to visualize how waves can interfere with eachother. Use the sliders to change the frequency and amplitude of the blue and orange waves and see how it changes the green interference wave, the wave created by adding the blue and orange waves together.

In [ ]:
# @title Interference Simulator
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Time array for smooth curves
t = np.linspace(0, 2*np.pi, 2000)

def plot_interference(f1=1.0, f2=1.0, phase_diff=0.0, amp1=1.0, amp2=1.0):

    y1 = amp1 * np.sin(2 * np.pi * f1 * t)
    y2 = amp2 * np.sin(2 * np.pi * f2 * t + phase_diff)
    y_sum = y1 + y2

    ymax = amp1 + amp2
    ymin = -(amp1 + amp2)

    plt.figure(figsize=(10, 5))
    plt.plot(t, y1, label='Wave 1', color='blue', alpha=0.7)
    plt.plot(t, y2, label='Wave 2', color='orange', alpha=0.7)
    plt.plot(t, y_sum, label='Interference Wave', color='green', linewidth=2)
    plt.title('Interference of Two Sine Waves', fontsize=16)
    plt.xlabel('Time', fontsize=14)
    plt.ylabel('Amplitude', fontsize=14)
    plt.ylim(ymin, ymax)
    plt.legend(loc='upper right', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.show()

# Interactive sliders with step=0.5
interact(plot_interference,
         f1=FloatSlider(value=-0.5, min=-2, max=2, step=0.5, description='Freq 1'),
         f2=FloatSlider(value=0.5, min=-2, max=2, step=0.5, description='Freq 2'),
         phase_diff=FloatSlider(value=0.0, min=0, max=2*np.pi, step=0.5, description='Phase Δ'),
         amp1=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.5, description='Amp 1'),
         amp2=FloatSlider(value=1.0, min=0.5, max=2.0, step=0.5, description='Amp 2')
        )


An example of interference using quantum computing ideas is the following, if we have the state $|\psi\rangle = \frac{1}{\sqrt{2}}|0\rangle+|1\rangle$
and we apply the Hadamard gate to it we get this result

\begin{align}
H|{\psi}⟩
&= H \left( \frac{1}{\sqrt{2}} (|{0}⟩ + |{1}⟩) \right) \\
&= \frac{1}{\sqrt{2}} \left( H|{0}⟩ + H|{1}⟩ \right) \\
&= \frac{1}{2} \left[ (|{0}⟩+|{1}⟩) + (|{0}⟩-|{1}⟩) \right] \\
&= \frac{1}{2} (2|{0}⟩) \\
&= |{0}⟩\
\end{align}


Looking at line 4, we can see there is a +$|1\rangle$ from the original 0 state and a -$|1\rangle$ from the original 1 state which cancel out since they add opposite signs. The probablitly amplitudes of the $|1\rangle$ state interfered destructively which in this case caused the $|1\rangle$ to be completely removed from the system. On the other hand, we can see there is a +$|0\rangle$ from the original 0 state and a +$|0\rangle$ from the original 1 state. Since they have the same sign they constructivly interfere and in this case they cause the system to be measured with 100% probabiltiy in the 0 state.


Conceptual check in:
What do you think will be left over after this operation
$H|\psi\rangle = H\left(\frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)\right)$

<details>
  <summary>Click to expand the answer<br></summary>

\begin{align}
H|{\psi}⟩
&= H \left( \frac{1}{\sqrt{2}} (|{0}⟩ - |{1}⟩) \right) \\
&= \frac{1}{\sqrt{2}} \left( H|{0}⟩ - H|{1}⟩ \right) \\
&= \frac{1}{2} \left[ (|{0}⟩+|{1}⟩) - (|{0}⟩-|{1}⟩) \right] \\
&= \frac{1}{2} (2|{1}⟩) \\
&= |{1}⟩\
\end{align}


</details>


In [ ]:
# @title Question 4
ask_mcq(
    "q4",
    "What is the effect of applying a Hadamard gate to |0⟩?",
    [
        "Leaves it unchanged",
        "Collapses it to |1⟩",
        "Creates an equal superposition of |0⟩ and |1⟩",
        "Measures the qubit"
    ]
    , correct = "Creates an equal superposition of |0⟩ and |1⟩"
)


In [ ]:
# @title Question 5
ask_mcq(
    "q5",
    "Why don’t we usually measure the qubits immediately after creating superposition?",
    [
        "Measurement would collapse the state too early",
        "Measurement makes the algorithm faster",
        "Measurement increases entanglement",
        "It doesn’t matter"
    ],
    correct = "Measurement would collapse the state too early"
)


## Grover's Algorithm Overview

Imagine you are searching for a single correct item hidden somewhere in a huge, unsorted list.
On a classical computer, you would check items one by one.
On a quantum computer, Grover’s Algorithm lets you “look at all items at once” and gradually boost the probability of finding the correct one.

Grover’s Algorithm works in three main ideas:

**Superposition** – look everywhere at once

**Oracle** – recognize the correct answer
# Grover’s Algorithm – Intuition Walkthrough

Imagine you are searching for a single correct item hidden somewhere in a huge, unsorted list.  
On a classical computer, you would check items one by one.  
On a quantum computer, Grover’s Algorithm lets you “look at all items at once” and gradually boost the probability of finding the correct one.

Grover’s Algorithm works in three main ideas:

- **Superposition** – look everywhere at once  
- **Oracle** – recognize the correct answer  
- **Amplitude Amplification** – make the correct answer stand out  


---

## Superposition

Grover’s Algorithm starts by putting the system into a superposition of all possible answers.


---

## Oracle — Marking the Correct Answer

The oracle is a special quantum operation that recognizes the correct answer and “Marks” it without telling us what it is directly.  

Instead of pointing at the correct answer, the oracle flips the sign (phase) of the correct state’s amplitude.  

The oracle does not reveal the answer, it only tags the correct state so it can be amplified later.


---

## Amplitude Amplification — Making the Answer Stand Out

After marking the correct answer, Grover’s Algorithm boosts the probability of measuring it and shrinks the probabilities of all others.  

When we measure a system it collapses to one of the states and the probability of collapsing to each state is the amplitude squared so the higher the amplitude is the higher chance the system has to collapse to that state.

This is done through a clever “reflection” process which is repeated multiple times:

- Reflect amplitudes around their average  
- The marked state grows larger each time  
- Others shrink slightly  


After enough repetitions the correct answer is much more likely to be measured so measuring the system gives the right answer with high probability.


---

## Why Grover’s Algorithm Is Powerful

For an unsorted list of length **N**, instead of a classic search algorithm taking **O(N)** to search through it, Grover’s Algorithm takes approximately: $
O(\sqrt{N})
$

Because Binary Search also has **O(log N)** run time for structured sorted data, it is actually better to use that than Grover's Algorithm if the data is sorted.  

Grover's is only better on unsorted data where it out performs the best classical searching algorithm.

**Amplitude Amplification** – make the correct answer stand out  

**Superposition**

Grover’s Algorithm starts by putting the system into a superposition of all possible answers.


**Oracle** Marking the Correct Answer

The oracle is a special quantum operation that recognizes the correct answer and “Marks” it without telling us what it is directly.
Instead of pointing at the correct answer, the oracle flips the sign (phase) of the correct state’s amplitude.
The oracle does not reveal the answer, it only tags the correct state so it can be amplified later


**Amplitude Amplification** Making the Answer Stand Out

After marking the correct answer, Grover’s Algorithm boosts the probability of measuring it and shrinks the probabilities of all others. When we measure a system it collapses to one of the states and the probability of collapsing to each state is the amplitude squared so the higher the amplitude is the higher chance the system has to collapse to that state.
This is done through a clever “reflection” process which is repeated multiple times:
Reflect amplitudes around their average
The marked state grows larger each time
Others shrink slightly


After enough repetitions the correct answer is much more likely to be measured so measuring the system gives the right answer with high probability


**Why Grover’s Algorithm Is Powerful**

For an unsorted list of length N instead of a classic search algorithm taking O(N) to search through it, Grover’s Algorithm takes ~O(√N)

Because Binary Search also has O(log N) run time for structured sorted data, it is actually better to use that than Grover's Algorithm if the data is sorted. Grover's is only better on unsorted data where it out performs the best classical searching alogrithm.

In [ ]:
# @title Question 6
ask_mcq(
    "q6",
    "Which classical algorithm does Grover’s replace?",
    [
        "Binary search",
        "Hash-based search",
        "Linear (brute-force) search",
        "None of them"
    ],
    correct = "Linear (brute-force) search"
)


In [ ]:
# @title Question 7
ask_mcq(
    "q7",
    "For a list of size N, how does Grover’s algorithm compare to classical brute-force search?",
    [
        "Both take O(N) steps",
        "Grover’s takes O(log N) steps",
        "Grover’s takes O(√N) steps",
        "Grover’s is constant time"
    ],
    correct = "Grover’s takes O(√N) steps"
)


In [ ]:
# @title Question 8
ask_mcq(
    "q8",
    "In which situation would Grover’s algorithm be most useful?",
    [
        "Searching a sorted list",
        "Searching an unsorted list with no structure",
        "Searching a hash table",
        "Searching a very small list"
    ], correct="Searching an unsorted list with no structure",
)


In [ ]:
# @title Question 9
ask_mcq(
    "q9",
    "Why does Grover’s algorithm provide a speedup over brute-force search?",
    [
        "It exploits quantum superposition and amplitude amplification",
        "It sorts the data first",
        "It uses classical hashing",
        "I’m not sure"
    ], correct = "It exploits quantum superposition and amplitude amplification"
)


In [ ]:
# @title Question 10
ask_mcq(
    "q10",
    "Which statement is true?",
    [
        "Grover’s algorithm works best on structured data",
        "Grover’s algorithm works best on unstructured data",
        "Grover’s algorithm replaces binary search",
        "None of the above"
    ],
    correct="Grover’s algorithm works best on unstructured data"
)


## Running Grover's Algorithm on AER Simulator

To run Grover's Algorithm we will first go through running it on Qiskit's AER simulator. Qiskit is a python library created by IBM. Its used to create quantum circuits and algorithms both on simulators and real quantum computers. In this activity we will use thier AER simulator. The AER simulator mimics a quantum circuit backend and this will allow us to create quantum circuits and see what they do without having to use a real quantum machine.


In [ ]:
# @title Qiskit imports
#run this if you have never installed qiskit before
!pip install -q qiskit
!pip install -q qiskit-aer
!pip install -q pylatexenc
import qiskit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_state_city
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline



Now we can start writing some actual quantum code!
As was stated earlier, one of the most often used quantum gates is the hadamard gate. Lets go through an example where we create a quantum ciruit with 1 qubit, apply the hadamard gate to it, and plot the probabilites associated with that qubit.

In [ ]:
n = 1 # n is the number of qubits in our quantum circuit


cir = qiskit.QuantumCircuit(n) # create the quantum circuit

cir.h(0) # apply the h (hadarmard) gate to qubit 0, the only qubit in our system


# plot the current amplitudes
cir.save_statevector()
sim = AerSimulator(method="statevector")
result = sim.run(cir).result()
state = result.get_statevector()
labels = [format(i, f'0{n}b') for i in range(2**n)]
probs = np.abs(state)**2

plt.bar(labels, probs)
plt.xlabel("Basis states")
plt.ylabel("Probability")
plt.title("Probability distribution after Hadamard gate")
plt.show()


Along the 'x' axis we can see all the possible states the quantumm system can be in, and along the 'y' axis we can see the probability of measuring the system to be in each of those states.

In [ ]:
#Optional draw the circuit
cir.draw(output="mpl")

Now you have seen your first quantum circuit! We have succesfully put qubit 0 into superposition between the 0 and 1 state. Putting qubits in superposition is the first step of many quantum algorithms including Grover's search algorithm. Usually however, we are searching through a list that is longer than can be represented with 1 qubit, lets create a bigger quantum circuit next.

In [ ]:
n = 3 # n is the number of qubits in our quantum circuit
#You are more than welcome to change the value of n to see how it changed things
#For the test cases below it assumes the value of n is 3


### START YOUR TURN

# TODO step 1: create the quantum circuit called "circ"

# TODO step 2: apply the hadamard gate to all qubits

### END YOUR TURN

# plot the current amplitudes
circ.save_statevector()
sim = AerSimulator(method="statevector")
result = sim.run(circ).result()
state = result.get_statevector()
labels = [format(i, f'0{n}b') for i in range(2**n)]
probs = np.abs(state)**2

plt.bar(labels, probs)
plt.xlabel("Basis states")
plt.ylabel("Probability")
plt.title("Probability distribution after Hadamard gate")
plt.show()

In [ ]:
# @title Test Cell
# HIDDEN TEST CELL

try:
    # Check if 'circ' exists
    assert 'circ' in globals(), "You must define a circuit named 'circ'"

    # Check number of qubits
    assert circ.num_qubits == n, f"Your circuit should have {n} qubits"

    # Simulate the statevector
    sim = AerSimulator(method="statevector")
    result = sim.run(circ).result()
    state = result.get_statevector()
    probs = np.abs(state)**2

    # Expected equal superposition probabilities
    expected_prob = 1 / 2**n
    tol = 1e-10

    # Check all probabilities are equal
    assert all(np.abs(p - expected_prob) < tol for p in probs), "Probabilities do not match equal superposition. Did you apply H to all qubits?"

except AssertionError as e:
    print("❌ Test failed:", e)
else:
    print("✅ All tests passed: Hadamard applied to all qubits, equal superposition achieved.")


In [ ]:
#Optional draw the circuit
cir.draw(output="mpl")

Now we will start working on the full version of Grover's search. We will define a target state that we are searching for, feel free to change it to be anything you like just make sure it stays length 3.
As stated before, Grover's search starts by putting all the qubits in equal superposition. Copy your code from above to do so.

In [ ]:
target = "110" #feel free to change this
n = len(target)
if (n != 3):
  raise ValueError ("target length has to be equal to 3")
target = target[::-1] # reverse the bit string

cir = qiskit.QuantumCircuit(n)

# put the ciruit in a superpositon of all possible states

### TODO PASTE YOUR CODE HERE
#TODO change circuit name to be 'cir' now


# plot the current amplitudes
cir.save_statevector()
sim = AerSimulator(method="statevector")
result = sim.run(cir).result()
state = result.get_statevector()
labels = [format(i, f'0{n}b') for i in range(2**n)]

plt.bar(labels, state)
plt.xlabel("Basis states")
plt.ylabel("Amplitude")
plt.title("Amplitude distribution of equal super position")
plt.show()


This time we have plotted the amplitude of each possible state. We can see that the amplidute and therefore probability to be in each of those states is equal.

In [ ]:
#Optional draw the circuit
cir.draw(output="mpl")


The next step of Grover's search is the Oracle. On a high level, the Oracle works by flipping the amplitude of the target state to be negative. In this way it "marks" the target state for the next step of the algorithm.



A more step by step explaination of the Oracle is as follows:


**Step 1: Map the target state**

We temporarily flip qubits so that the target state becomes all ones.  
- Example: if the target is |101⟩, we flip the qubits with 0 → 1 so it becomes |111⟩.  
- This allows a simple operation to target only that state.


**Step 2: Flip the amplitude**

We flip the amplitude of the |111⟩ state to be negative.  
- All other states remain unchanged.  
- This “tags” the target state for amplification.

**Step 3: Undo the flips**

We reverse the qubit flips from Step 2 so that the negative amplitude is back at the original target state.  
- All other states are still unchanged.


**Step 4: Result**

- The target state now has a negative amplitude  
- Other states remain the same  
- This sets up the next step of Grover's Algorithm: **amplitude amplification**, which increases the probability of measuring the correct answer.


In [ ]:
# The circuit is in a superposition of all states
# before being passed to Oracle

def Oracle(circuit):
    # step 1: Map the target state
    for i in range (n): # loop over all qubits
        if target[i] == '0': #if the target state has a '0'
            circuit.x(i) #apply a X (NOT) gate to flip it to be a 1

    #step 2: flip the amplitude
    # does a phase flip on the '111' state to change
    # its amplidute to be negative
    circuit.h(n-1)
    circuit.mcx(list(range(n-1)), n-1)
    circuit.h(n-1)


    #step 3: Undo the flip
    # undoes the change from the first loop
    # now the target state has negative aplitude
    # and is "marked" which is what we want
    for i in range (n):
        if target[i] == '0':
            circuit.x(i)


Lets create the ciruit again and apply the oracle to see how the amplitudes change

In [ ]:

cir = qiskit.QuantumCircuit(n)

# put the circuit in a superpositon of all possible states

### TODO PASTE YOUR CODE HERE TO PUT CIRCUIT IN SUPERPOSITION


# Apply the Oracle
Oracle(cir)


# plot the current amplitudes
cir.save_statevector()
sim = AerSimulator(method="statevector")
result = sim.run(cir).result()
state = result.get_statevector()
labels = [format(i, f'0{n}b') for i in range(2**n)]

plt.bar(labels, state)
plt.xlabel("Basis states")
plt.ylabel("Amplitude")
plt.title("Amplitude distribution after Oracle")
plt.show()

As we can see, only the amplitude of the target state is changed and specifically it is fliped to be negative now.
While this is helpful to be able to "find" something we really want the probabilty of measuring that state to be relativly high so we get the correct answer. Lets try plotting the probabily of measuring each state.

In [ ]:
labels = [format(i, f'0{n}b') for i in range(2**n)]
probs = np.abs(state) ** 2
plt.bar(labels, probs)
plt.xlabel("Basis states")
plt.ylabel("Probability")
plt.title("Probability distribution after Oracle")
plt.show()

As we can see, even though the amplitude of the target state has been flipped to be negative the probability of it being measured is still the same as all the other states. This is because probabilty equals amplitude squared so just making the amplitude negative isn't enough, we need to do something else to make it more certain we will measure the right result.

In [ ]:
#Optional draw the circuit
cir.draw(output="mpl")

In [ ]:
# @title Question 11
ask_mcq(
    "q11",
    "What role does the oracle play in Grover’s algorithm?",
    [
        "It sorts the search space",
        "It marks the correct state by changing its amplitude",
        "It measures the system",
        "It stores the database"
    ],
    correct = "It marks the correct state by changing its amplitude"
)


Next we will define the Diffusion step

The diffusion step in Grover’s algorithm is the part that increases the likelihood of measuring the correct answer. After the oracle marks the correct state by flipping its amplitude, the diffusion step reflects all amplitudes about their average value. This causes the marked state’s amplitude to increase while the amplitudes of the other states decrease slightly. Although no probabilities change directly during the oracle step, the diffusion step uses quantum interference to turn that phase difference into a higher measurement probability for the correct answer. Repeating the oracle and diffusion steps gradually amplifies the correct result.

**How the diffusion step uses interference**

In Grover’s algorithm, the oracle first flips the sign of the correct state's amplitude. Then the diffusion step figures out what the average amplitude is then reflects the amplitudes of the states around it causing the flipped state to constructively interfere and grow, while the others destructively interfere and shrink. In this way, interference systematically moves probability away from wrong answers and concentrates it on the correct one.

Mathmatically the Diffusion step calculates the average amplitude $\mu$. It then uses iterference to change every other amplitude of the system according to this formula $a_x = (2\mu-a_x$) where $a_x$ is the amplitude of the "$x$th" basis state. Because of this rotating around the average it causes the amplitudes to oscillate.  

In [ ]:
def Diffusion(circuit):

    #convert back to the original basis
    for i in range (n):
        circuit.h(i)

    #flip all the qbits
    for i in range (n):
        circuit.x(i)


    circuit.h(n-1)
    circuit.mcx(list(range(n-1)), n-1)
    circuit.h(n-1)

    #return circuit to correct basis
    for i in range (n):
        circuit.x(i)
    for i in range (n):
        circuit.h(i)

Once again we will recreate the circuit and plot how it changes after both the Oracle and Diffusion step have been applied. Going forward we will refer to this as a 'Grover iteration' meaning an application of the Oracle then Diffusion.

In [ ]:
cir = qiskit.QuantumCircuit(n)
# put the ciruit in a superpositon of all possible states
cir.h(range(n))
Oracle(cir)
Diffusion (cir)

# plot the current probabilites
cir.save_statevector()
sim = AerSimulator(method="statevector")
result = sim.run(cir).result()
state = result.get_statevector()
labels = [format(i, f'0{n}b') for i in range(2**n)]
probs = np.abs(state) ** 2

plt.bar(labels, probs)
plt.xlabel("Basis states")
plt.ylabel("Probability")
plt.title("Probability distribution after one Grover iteration")
plt.show()

The probabilty of measuring the target state is even higher now and we should feel pretty confident that we would get the correct result

In [ ]:
#Optional draw the circuit
cir.draw(output="mpl")

In [ ]:
# @title Question 12
ask_mcq(
    "q12",
    "What do you expect the probability of the correct answer to be after one Grover iteration?",
    [
        "Very low",
        "Higher than others",
        "Same as the others",
        "Exactly 1"
    ], correct= "Higher than others"
)


In [ ]:
# @title Question 13
ask_mcq(
    "q13",
    "Which best describes what applying the Oracle and Diffusion Step does?",
    [
        "Increases the probability of the correct answer",
        "Decreases the probability of incorrect answers",
        "All of the above"
    ],
    correct = "All of the above"
)


As we can see, the probability of the target state went up by a lot while the probabliy of the other states went down

In Grover's Algorithm, you usually repeat the process of applying the Oracle then diffusion step more than once
Lets calculate K, the amount of times to apply the 2 processes

In [ ]:
K = int(np.floor(np.pi / 4 * np.sqrt(2 ** n)))
print("K: ", K)

Now we can do a full run through of Grover's algorithm. Look at the comments below for where to write code and run the test cell below to see if it is correct.

In [ ]:

# --- build circuit step-by-step ---
cir = ### TODO replace this with code to create circuit with 'n' qubits


### TODO apply 'h' gates to put circuit in equal superposition


for _ in range(K):

    ### START YOUR TURN

    #TODO write code to apply the Oralce and Diffusion step here

    ### END YOUR TURN



# plot the current probabilites
cir.save_statevector()
sim = AerSimulator(method="statevector")
result = sim.run(cir).result()
state = result.get_statevector()
labels = [format(i, f'0{n}b') for i in range(2**n)]
probs = np.abs(state) ** 2

plt.bar(labels, probs)
plt.xlabel("Basis states")
plt.ylabel("Probability")
plt.title("Probability distribution")
plt.show()


In [ ]:
# @title TESTS CELL
# --- HIDDEN TEST CELL ---
try:
    # 1. Check that the circuit exists
    assert 'cir' in globals(), "Circuit 'cir' is not defined."

    # 2. Check number of qubits
    assert cir.num_qubits == n, f"Circuit should have {n} qubits, found {cir.num_qubits}."

    # 3. Check initial Hadamard gates (first n gates should be H)
    from qiskit.circuit.library.standard_gates import HGate
    hadamard_count = sum(1 for gate, qargs, cargs in cir.data[:n] if isinstance(gate, HGate))
    assert hadamard_count == n, f"Hadamard gates missing on some qubits. Expected {n}, found {hadamard_count}."

    # 4. Simulate statevector after Grover iterations


    sim = AerSimulator(method="statevector")
    result = sim.run(cir).result()
    state = result.get_statevector()

    # Compute probabilities
    probs = np.abs(state.data)**2

    # Find the state with maximum probability
    max_prob_index = np.argmax(probs)
    max_prob = probs[max_prob_index]

    # Check if max probability corresponds to target
    # Convert target list/string to index
    target = target[::-1]
    if isinstance(target, str):
        target_index = int(target, 2)
    else:
        target_index = int(''.join(str(b) for b in target), 2)

    assert max_prob_index == target_index, f"Maximum probability is not on the target state. Target index {target_index}, max prob index {max_prob_index}."
    assert max_prob > 0.5, f"Target state's probability should be high (>0.5), found {max_prob:.2f}."

except AssertionError as e:
    print("❌ Test failed:", e)
else:
    print("✅ All tests passed: circuit structure and Grover amplification look correct.")


Run the cell below to see an interative animation of how the probabiltes of each state change after each Grover iteration.

In [ ]:
# @title How probabilites change after each Grover iteration animation
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

sim = AerSimulator(method="statevector")

states = []

def get_state(circ):
    tmp = circ.copy()
    tmp.save_statevector()
    result = sim.run(tmp).result()
    return result.get_statevector()

# --- build circuit step-by-step ---
cir = QuantumCircuit(n)

# initial H layer
cir.h(range(n))
states.append(get_state(cir))

for _ in range(K):
    Oracle(cir)

    Diffusion(cir)
    states.append(get_state(cir))   # after diffusion


labels = [format(i, f'0{n}b') for i in range(2**n)]

fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(labels, np.zeros(len(labels)))

ax.set_ylim(0, 1)
ax.set_xlabel("Basis states")
ax.set_ylabel("Probability")

def update(frame):
    probs = np.abs(states[frame])**2
    for bar, p in zip(bars, probs):
        bar.set_height(p)
    ax.set_title(f"Grover Iteration {frame}")
    return bars

ani = FuncAnimation(fig, update, frames=len(states), interval=700)

plt.close()
HTML(ani.to_jshtml())


In [ ]:
# @title Question 14
ask_mcq(
    "q14",
    "After several Grover iterations, what do you observe in the probability distribution?",
    [
        "All states become equally likely",
        "The marked state’s probability increases relative to others",
        "Probabilities disappear",
        "The system becomes classical"
    ],
    correct = "The marked state’s probability increases relative to others"
)


After running the Oracle and Diffusion step twice we see that the probabiltiy of measuring the target state has gotten quite high but can we do better if we increase K? Lets see what happens if we run the circuit one more time with K being 5.

In [ ]:
K = 5

sim = AerSimulator(method="statevector")

states = []

def get_state(circ):
    tmp = circ.copy()
    tmp.save_statevector()
    result = sim.run(tmp).result()
    return result.get_statevector()

# --- build circuit step-by-step ---
cir = QuantumCircuit(n)

# initial H layer
cir.h(range(n))
states.append(get_state(cir))

for _ in range(K):
    Oracle(cir)

    Diffusion(cir)
    states.append(get_state(cir))   # after diffusion


labels = [format(i, f'0{n}b') for i in range(2**n)]

fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar(labels, np.zeros(len(labels)))

ax.set_ylim(0, 1)
ax.set_xlabel("Basis states")
ax.set_ylabel("Probability")

def update(frame):
    probs = np.abs(states[frame])**2
    for bar, p in zip(bars, probs):
        bar.set_height(p)
    ax.set_title(f"Step {frame}")
    return bars

ani = FuncAnimation(fig, update, frames=len(states), interval=700)

plt.close()
HTML(ani.to_jshtml())


As we can see the probabilites of measuring the other states has gone up not down, this is the opposite of what we want. You can keep running the code block above to see what happnes if we keep increasing K. Try and spot a pattern in the probabilty distrubution.

In [2]:
# @title Question 15
ask_mcq(
    "q15",
    "If the Oracle and Diffusion Step are run too many times, what happens?",
    [
        "The probability of the correct answer keeps increasing",
        "The probability oscillates and may decrease",
        "The answer becomes deterministic",
        "The circuit crashes"
    ], correct = "The probability oscillates and may decrease"
)


HTML(value="<b style='font-size:24px'>If the Oracle and Diffusion Step are run too many times, what happens?</…

RadioButtons(_dom_classes=('big-radio',), layout=Layout(width='auto'), options=('The probability of the correc…

Button(button_style='primary', description='Check answer', style=ButtonStyle())

Output()

HTML(value='\n    <style>\n    .big-radio label {\n        font-size: 20px !important;\n    }\n    </style>\n …

##### Do this after they complete the post survey if they have time and/or they are interested on learning how to run on a real quantum computer. Show them also how to get API key

Now that we have a general idea of how Grover's Algorithm works, lets try running it on a real IBM quantum computer

In [ ]:
!pip install -q qiskit_ibm_runtime

In [ ]:
#Initial set up imports
import qiskit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit_ibm_runtime import SamplerV2
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np

Link to tutorial to get API key:
https://docs.google.com/document/d/1NRANdujFTsbKeh9yVKl-w09chWuR4OLPGI-4_6g1xqU/edit?usp=drive_link

Link to IBM website:
https://quantum.cloud.ibm.com/

In [ ]:
#Link your IBM account
QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token="PASTE API TOKEN HERE",
    overwrite=True
)

Now we will create the circuit, this is mostly the same as what we did before, just some syntax differences.

In [ ]:
# Set up target and Quantum Circuit
target = "101"
target = target[::-1]
n = len(target)

qc = QuantumCircuit(n)
qc.h(range(n))

In [ ]:
#Define the Oracle for a real Quantum Computer
def Oracle_Real(circuit):
    for i in range(n):
        if target[i] == "0":
            circuit.x(i)
    circuit.h(n-1)
    circuit.mcx(list(range(n-1)), n-1)
    circuit.h(n-1)
    for i in range(n):
        if target[i] == "0":
            circuit.x(i)


In [ ]:
#Define Diffusion for a real Quantum Computer
def Diffusion_Real(circuit):
    for i in range(n):
        circuit.h(i)
    for i in range(n):
        circuit.x(i)
    circuit.h(n-1)
    circuit.mcx(list(range(n-1)), n-1)
    circuit.h(n-1)
    for i in range(n):
        circuit.x(i)
    for i in range(n):
        circuit.h(i)


In [ ]:
# Apply the Oracle and Diffusion step K times
k = int(np.floor(np.pi / 4 * np.sqrt(2 ** n)))
for _ in range(k):
    Oracle_Real(qc)
    Diffusion_Real(qc)

#measure the results
qc.measure_all()

Now that we have th circuit built lets find a real quantum computer to run it on. Be careful running the code block below too manhy times since it does use up quantum resources that have a limit for when they are free

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=n
)
print("Running on:", backend.name)


Now we can actually run the circuit. We will run it multiple times and plot what results we get

In [ ]:
qc_transpiled = transpile(qc, backend=backend, optimization_level=3)

sampler = SamplerV2(mode=backend)

# Submit the job and get results
job = sampler.run([qc_transpiled], shots=1024)
print(f"Job ID: {job.job_id}")
result = job.result()
counts = result[0].data.meas.get_counts()
print(f"Measurement counts: ", counts)


plot_histogram(counts)
plt.show()

As we can see the probablity of our target state is still much higher than the rest of the states.
Unlike with the AER simulator however, the other state probabilites are a little higher and more random. This is due to the fact that real quantum computers have noice or interference that changes the probabilites.